# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.8%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

## 1.準備

In [1]:
# omnicampus 実行用
!pip install ipywidgets


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cuda


# For Colab

In [ ]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

# For Local

In [2]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


In [3]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
#WORK_DIR = "/content/drive/MyDrive/weblab/DLBasics2025/Competition"
WORK_DIR = os.path.join(work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

c:\Users\dysk-\Desktop\Current task\EEG compe


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset

class ThingsEEGDataset(Dataset):
    def __init__(self, split: str, use_vit: bool = True):
        assert split in ["train", "val", "test"]
        self.split = split
        self.use_vit = use_vit

        self.X = np.load(f"data/{split}/eeg.npy").astype(np.float32)

        # trial-wise z-score
        self.X = (self.X - self.X.mean(axis=-1, keepdims=True)) / (
            self.X.std(axis=-1, keepdims=True) + 1e-6
        )

        self.X = np.clip(self.X, -5, 5)

        self.subject = np.load(f"data/{split}/subject_idxs.npy").astype(np.int64)
        self.subject = self.subject - 1

        if split != "test":
            self.y = np.load(f"data/{split}/labels.npy").astype(np.int64)
        else:
            self.y = None

        if use_vit and split != "test":
            self.vit = np.load(f"data/{split}/vit_features.npy").astype(np.float32)

            # ViT特徴は方向情報を使いたいのでL2 normalize
            self.vit = self.vit / (np.linalg.norm(self.vit, axis=1, keepdims=True) + 1e-6)
        else:
            self.vit = None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        subject = torch.tensor(self.subject[idx], dtype=torch.long)

        if self.split == "test":
            return x, subject

        y = torch.tensor(self.y[idx], dtype=torch.long)

        if self.use_vit:
            vit = torch.tensor(self.vit[idx], dtype=torch.float32)
            return x, subject, y, vit

        return x, subject, y

# 2.5 Load Config file

In [16]:
del run_dir

In [5]:


from pathlib import Path
from datetime import datetime
import json
import shutil

# ===== 読み込むconfigを指定 =====
#CONFIG_PATH =  Path("configs/baseline.json")
#CONFIG_PATH =  Path("configs/clip_m5_5.json")
#CONFIG_PATH =  Path("configs/baseline_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip_SubjectEmbedding.json")
CONFIG_PATH =  Path("configs/b_baseline_eeg_to_vit_mse_cos.json")


print(f"Loading config from: {CONFIG_PATH}")
#CONFIG_PATH = work_dir + CONFIG_PATH

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

# ===== configから変数に反映 =====
RUN_NAME = config["run_name"]
seed = config["seed"]
lr = config["lr"]
batch_size = config["batch_size"]
epochs = config["epochs"]
model_name = config["model_name"]
optimizer_name = config["optimizer"]
scheduler_name = config["scheduler"]

# ===== 保存先作成 =====
if "run_dir" not in globals():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    run_dir = Path("outputs") / f"{timestamp}_{RUN_NAME}"
    run_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(CONFIG_PATH, run_dir / "config.json")

print(f"Run directory: {run_dir}")

Loading config from: configs\b_baseline_eeg_to_vit_mse_cos.json
Run directory: outputs\20260610_0437_b_baseline_eeg_to_vit_mse_cos


# Load image_features data

In [6]:
from pathlib import Path
import numpy as np

feature_path = work_dir + "/data/features/vit_image_features.npy"
path_txt = work_dir + "/data/features/vit_image_paths.txt"
print(feature_path)


features = np.load(feature_path)

with open(path_txt) as f:
    feature_paths = [p.strip() for p in f.readlines()]

print(features.shape)
print(len(feature_paths))
print(feature_paths[0])

c:\Users\dysk-\Desktop\Current task\EEG compe/data/features/vit_image_features.npy
(5940, 768)
5940
00001_aardvark/aardvark_01b.jpg


In [7]:
# path -> feature の辞書
feature_dict = {
    p: feat
    for p, feat in zip(feature_paths, features)
}

def make_trial_image_features(split):
    path_file = work_dir + f"/data/{split}/image_paths.txt"

    with open(path_file) as f:
        trial_paths = [p.strip() for p in f.readlines()]

    trial_features = np.stack([
        feature_dict[p]
        for p in trial_paths
    ])

    return trial_features

train_img_feats = make_trial_image_features("train")
val_img_feats = make_trial_image_features("val")

print(train_img_feats.shape)
print(val_img_feats.shape)

(118800, 768)
(59400, 768)


In [8]:
np.save(work_dir + "/data/train/vit_features.npy", train_img_feats)
np.save(work_dir + "/data/val/vit_features.npy", val_img_feats)

## 3.ベースラインモデル

In [ ]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size: int = 3,
        p_drop: float = 0.1,
    ) -> None:
        super().__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim

        self.conv0 = nn.Conv1d(in_dim, out_dim, kernel_size, padding="same")
        self.conv1 = nn.Conv1d(out_dim, out_dim, kernel_size, padding="same")
        # self.conv2 = nn.Conv1d(out_dim, out_dim, kernel_size) # , padding="same")

        self.batchnorm0 = nn.BatchNorm1d(num_features=out_dim)
        self.batchnorm1 = nn.BatchNorm1d(num_features=out_dim)

        self.dropout = nn.Dropout(p_drop)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if self.in_dim == self.out_dim:
            X = self.conv0(X) + X  # skip connection
        else:
            X = self.conv0(X)

        X = F.gelu(self.batchnorm0(X))

        X = self.conv1(X) + X  # skip connection
        X = F.gelu(self.batchnorm1(X))

        # X = self.conv2(X)
        # X = F.glu(X, dim=-2)

        return self.dropout(X)


class BasicConvClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        seq_len: int,
        in_channels: int,
        hid_dim: int = 128
    ) -> None:
        super().__init__()

        self.blocks = nn.Sequential(
            ConvBlock(in_channels, hid_dim),
            ConvBlock(hid_dim, hid_dim),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            Rearrange("b d 1 -> b d"),
            nn.Linear(hid_dim, num_classes),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """_summary_
        Args:
            X ( b, c, t ): _description_
        Returns:
            X ( b, num_classes ): _description_
        """
        X = self.blocks(X)

        return self.head(X)
    


class EEGNetClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_channels: int,
        seq_len: int,
        F1: int = 32,
        D: int = 2,
        F2: int = 64,
        dropout: float = 0.5,
        subject_emb_dim: int = 16,
        num_subjects: int = 10,
    ):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 15), padding=(0, 7), bias=False),
            nn.BatchNorm2d(F1),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 15), padding=(0, 7),
                      groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        with torch.no_grad():
            dummy = torch.zeros(1, num_channels, seq_len)
            feat = self._forward_features(dummy)
            feat_dim = feat.shape[1]

        self.classifier = nn.Linear(feat_dim + subject_emb_dim, num_classes)

    def _forward_features(self, x):
        x = x.unsqueeze(1)  # (batch, 1, channels, time)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return x

    def forward(self, x, subject_idxs):
        x = self._forward_features(x)
        subject_emb = self.subject_embedding(subject_idxs)
        x = torch.cat([x, subject_emb], dim=1)
        return self.classifier(x)


import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNetEncoder(nn.Module):
    def __init__(self, num_channels=17, num_times=100, dropout=0.25):
        super().__init__()

        F1 = 64
        D = 2
        F2 = F1 * D

        self.net = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 25), padding=(0, 12), bias=False),
            nn.BatchNorm2d(F1),

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(num_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),

            nn.Conv2d(
                F1 * D,
                F1 * D,
                kernel_size=(1, 15),
                padding=(0, 7),
                groups=F1 * D,
                bias=False
            ),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, num_channels, num_times)
            out = self.net(dummy)
            self.out_dim = out.flatten(1).shape[1]

    def forward(self, x):
        x = x.unsqueeze(1)
        h = self.net(x)
        h = h.flatten(1)
        return h


class EEGToViTBaseline(nn.Module):
    def __init__(self, num_classes=5, num_subjects=10, subject_dim=16, vit_dim=768):
        super().__init__()

        self.encoder = EEGNetEncoder()
        self.subject_emb = nn.Embedding(num_subjects, subject_dim)

        hidden_dim = self.encoder.out_dim + subject_dim

        self.vit_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, vit_dim),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def encode(self, x, subject):
        h = self.encoder(x)
        s = self.subject_emb(subject)
        h = torch.cat([h, s], dim=1)
        return h

    def forward_vit(self, x, subject):
        h = self.encode(x, subject)
        z = self.vit_head(h)
        z = F.normalize(z, dim=1)
        return z

    def forward_cls(self, x, subject):
        h = self.encode(x, subject)
        logits = self.classifier(h)
        return logits

# Set Seed

In [9]:
import random
import numpy as np
import torch

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(seed)

## 4.訓練実行

In [7]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim

train_ds = ThingsEEGDataset("train", use_vit=True)
val_ds = ThingsEEGDataset("val", use_vit=True)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)

model = EEGToViTBaseline().to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

def mse_cos_loss(pred, target, alpha=0.5):
    pred = F.normalize(pred, dim=1)
    target = F.normalize(target, dim=1)

    mse = F.mse_loss(pred, target)
    cos = 1.0 - F.cosine_similarity(pred, target, dim=1).mean()

    loss = alpha * mse + (1.0 - alpha) * cos
    return loss, mse.detach(), cos.detach()

best_val_loss = float("inf")

for epoch in range(30):
    model.train()
    train_loss = 0.0

    for x, subject, y, vit in tqdm(train_loader, desc=f"pretrain {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        vit = vit.to(device)

        optimizer.zero_grad()

        pred = model.forward_vit(x, subject)
        loss, mse, cos = mse_cos_loss(pred, vit, alpha=0.5)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    scheduler.step()
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    val_cos_sim = 0.0

    with torch.no_grad():
        for x, subject, y, vit in val_loader:
            x = x.to(device)
            subject = subject.to(device)
            vit = vit.to(device)

            pred = model.forward_vit(x, subject)
            loss, mse, cos = mse_cos_loss(pred, vit, alpha=0.5)

            cos_sim = F.cosine_similarity(
                F.normalize(pred, dim=1),
                F.normalize(vit, dim=1),
                dim=1
            ).mean()

            val_loss += loss.item() * x.size(0)
            val_cos_sim += cos_sim.item() * x.size(0)

    val_loss /= len(val_ds)
    val_cos_sim /= len(val_ds)

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | "
        f"val_loss={val_loss:.5f} | "
        f"val_cos_sim={val_cos_sim:.5f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "model_b_pretrained.pt")
        print("saved: model_b_pretrained.pt")

pretrain 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=0.42207 | val_loss=0.41703 | val_cos_sim=0.16810
saved: model_b_pretrained.pt


pretrain 2:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 02 | train_loss=0.41641 | val_loss=0.41475 | val_cos_sim=0.17266
saved: model_b_pretrained.pt


pretrain 3:   0%|          | 0/465 [00:00<?, ?it/s]

...

epoch 28 | train_loss=0.39875 | val_loss=0.40743 | val_cos_sim=0.18726


pretrain 29:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 29 | train_loss=0.39868 | val_loss=0.40740 | val_cos_sim=0.18731


pretrain 30:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 30 | train_loss=0.39873 | val_loss=0.40738 | val_cos_sim=0.18735
saved: model_b_pretrained.pt


In [8]:
train_ds_ft = ThingsEEGDataset("train", use_vit=False)
val_ds_ft = ThingsEEGDataset("val", use_vit=False)

train_loader_ft = DataLoader(train_ds_ft, batch_size=256, shuffle=True, num_workers=0)
val_loader_ft = DataLoader(val_ds_ft, batch_size=512, shuffle=False, num_workers=0)

model = EEGToViTBaseline().to(device)
model.load_state_dict(torch.load("model_b_pretrained.pt", map_location=device))

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 3e-4},
        {"params": model.subject_emb.parameters(), "lr": 3e-4},
        {"params": model.classifier.parameters(), "lr": 1e-3},
    ],
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_acc = 0.0

for epoch in range(50):
    model.train()
    train_loss = 0.0
    train_correct = 0

    for x, subject, y in tqdm(train_loader_ft, desc=f"finetune {epoch+1}"):
        x = x.to(device)
        subject = subject.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model.forward_cls(x, subject)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(dim=1) == y).sum().item()

    scheduler.step()

    train_loss /= len(train_ds_ft)
    train_acc = train_correct / len(train_ds_ft)

    model.eval()
    val_loss = 0.0
    val_correct = 0

    with torch.no_grad():
        for x, subject, y in val_loader_ft:
            x = x.to(device)
            subject = subject.to(device)
            y = y.to(device)

            logits = model.forward_cls(x, subject)
            loss = criterion(logits, y)

            val_loss += loss.item() * x.size(0)
            val_correct += (logits.argmax(dim=1) == y).sum().item()

    val_loss /= len(val_ds_ft)
    val_acc = val_correct / len(val_ds_ft)

    print(
        f"epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.5f} | train_acc={train_acc:.5f} | "
        f"val_loss={val_loss:.5f} | val_acc={val_acc:.5f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "model_b_finetuned_best.pt")
        print(f"saved: model_b_finetuned_best.pt | val_acc={best_val_acc:.5f}")

C:\Users\dysk-\AppData\Local\Temp\ipykernel_39600\1999211025.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model_b_pretrained.pt", ma

finetune 1:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 01 | train_loss=1.35281 | train_acc=0.47730 | val_loss=1.30797 | val_acc=0.49968
saved: model_b_finetuned_best.pt | val_acc=0.49968


finetune 2:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 02 | train_loss=1.31121 | train_acc=0.49900 | val_loss=1.29827 | val_acc=0.50091
saved: model_b_finetuned_best.pt | val_acc=0.50091


...

epoch 48 | train_loss=1.15114 | train_acc=0.57331 | val_loss=1.25929 | val_acc=0.52276


finetune 49:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 49 | train_loss=1.15047 | train_acc=0.57662 | val_loss=1.25904 | val_acc=0.52379


finetune 50:   0%|          | 0/465 [00:00<?, ?it/s]

epoch 50 | train_loss=1.15051 | train_acc=0.57444 | val_loss=1.25901 | val_acc=0.52389


## 5.評価

In [9]:
test_ds = ThingsEEGDataset("test", use_vit=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)

model = EEGToViTBaseline().to(device)
model.load_state_dict(torch.load("model_b_finetuned_best.pt", map_location=device))
model.eval()

all_probs = []

with torch.no_grad():
    for x, subject in tqdm(test_loader, desc="predict"):
        x = x.to(device)
        subject = subject.to(device)

        logits = model.forward_cls(x, subject)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
y_pred = all_probs.argmax(axis=1)

# 提出用は必ず (N, 5) の確率配列
np.save("submission.npy", all_probs)

# 確認用にラベルも保存
np.save("y_pred.npy", y_pred)
np.save("probs_f1.npy", all_probs)

print("submission:", all_probs.shape)
print("y_pred:", y_pred.shape)
print("row sum:", all_probs.sum(axis=1)[:5])

C:\Users\dysk-\AppData\Local\Temp\ipykernel_39600\971787966.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model_b_finetuned_best.pt",

predict:   0%|          | 0/117 [00:00<?, ?it/s]

submission: (59400, 5)
y_pred: (59400,)
row sum: [1.         1.         1.         1.         0.99999994]


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [10]:
from zipfile import ZipFile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = run_dir / f"{timestamp}_submission.zip"

submission_path = run_dir / "submission.npy"
model_path = run_dir / "model_best.pt"
notebook_path = Path(work_dir) / "notebooks" / "DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile(zip_name, "w") as zf:
    zf.write(submission_path, arcname="submission.npy")
    zf.write(model_path, arcname="model_best.pt")
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb")

print(f"Created: {zip_name}")

with ZipFile(zip_name, "r") as zf:
    print(zf.namelist())

Created: outputs\20260610_0437_b_baseline_eeg_to_vit_mse_cos\20260610_0502_submission.zip
['submission.npy', 'model_best.pt', 'DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb']
